#### pandasなどのimport、解析前に環境設定

In [ ]:
import pandas as pd
from Bio import SeqIO, Entrez
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
mpl.rcParams['svg.fonttype'] = 'none'
# Path to your Arial.ttf file
arial_font_path = '/Users/izumikida/fonts/Arial.ttf'
# Create a FontProperties object for the Arial font
arial_font = FontProperties(fname=arial_font_path)
mpl.rcParams['font.family'] = arial_font.get_name()
# import matplotlib.pyplot as plt
# import seaborn as sns
import numpy as np
import time
import subprocess

In [4]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [7]:
# ORFのpositionを定義する関数 parse_positions
def parse_positions(pos_file):
    ORFs = {}
    with open(pos_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:  # skip empty line
                continue
            parts = line.split(',')
            # parts[0] = ORF name, parts[1] = start pos, parts[2] = end pos
            orfname = parts[0]
            start = int(parts[1])
            end = int(parts[2])
            ORFs[orfname] = [start, end]
    return ORFs
FCoV_pos_path = '/Users/izumikida/FCoV-evolution/FCoV1_HLJHRB201610_ORF_pos.csv'
FCoV_pos = parse_positions(FCoV_pos_path)
FCoV_pos

{'Orf1ab': [382, 20471],
 'S': [20468, 24877],
 'Orf3a': [24889, 25101],
 'Orf3b': [25049, 25270],
 'Orf3c': [25267, 25980],
 'E': [25967, 26215],
 'M': [26226, 27014],
 'N': [27027, 28160],
 'Orf7a': [28165, 28470],
 'Orf7b': [28475, 29095]}

In [8]:
#get the alignment positions based on one ref's pos
def al_pos(positions, al_file, ref):

    records = SeqIO.to_dict(SeqIO.parse(al_file, 'fasta'))
    numgaps = [str(records[n].seq).count('-') for n in list(records)]
    if len(ref) == 0:
        ref = list(records)[numgaps.index(min(numgaps))]
    refseqone = str(records[ref].seq)

    map = {}
    c = 0
    for i in range(len(refseqone)):
        if str(c) not in list(map):
            map.update({str(c):[i]})
        else:
            map[str(c)].append(i)
        if refseqone[i] != '-':
            c = c +1

    finalpos = [max(map[str(x)]) for x in positions]
    
    return finalpos


ORFs = {}
for name, coords in FCoV_pos.items():
    ORFs[name] = al_pos(coords, 'FCoV1_full_nt_ginsi_Izumi_250824.fasta', 'KY566209.1')
print(ORFs)

{'Orf1ab': [385, 20558], 'S': [20555, 25034], 'Orf3a': [25046, 25258], 'Orf3b': [25206, 25427], 'Orf3c': [25424, 26150], 'E': [26137, 26385], 'M': [26396, 27209], 'N': [27222, 28368], 'Orf7a': [28373, 28678], 'Orf7b': [28683, 29306]}


ORF毎にaliginment後のpositionで切り分ける　ATGで始まりが揃っているかと終わりが終始コドンで揃っているか確認

In [9]:
def extract_ORFs(orfd, wg_al_file, pr_name, ungap = 'Y', translate = 'Y'):

    recdict = SeqIO.to_dict(SeqIO.parse(wg_al_file, 'fasta'))

    for orf in list(orfd):
        
        # refseq = str(recdict[ref].seq)
        b = orfd[orf]
        start = b[0]-1
        end = b[1]
        # tp = 0
        # alp = 0
        # alstart = 0
        # alend = 0
        # for n in refseq:
                # if tp == start:
                        # alstart = alp
                # if tp == end:
                        # alend = alp
                # alp = alp+1
                # if n != '-':
                        # tp = tp+1
        
        alfile = 'al_%s_%s_nt_prelim.fas'%(pr_name, orf)
        ungapfile = '%s_%s_nt.fas'%(pr_name,orf)
        aafile = '%s_%s_aa.fas'%(pr_name, orf)
        
        fasta = ''
        ungapfasta = ''
        aafasta = ''    
        
        for virname in list(recdict):            
            # thisseq = str(recdict[virname].seq)[alstart:alend]
            thisseq = str(recdict[virname].seq)[start:end]
            fasta = fasta + '>%s\n%s\n'%(virname, thisseq)
            
            if ungap == 'Y':
                ungapseq = thisseq.replace('-', '')                
                ungapfasta = ungapfasta + '>%s\n%s\n'%(virname, ungapseq)
                
            if translate == 'Y':
                aaseq = Seq(thisseq.replace('-', '')).translate()         
                aafasta = aafasta + '>%s\n%s\n'%(virname, str(aaseq))
                    
        with open(alfile, 'w+') as out:
            out.write(fasta[:-1])            
        with open(ungapfile, 'w+') as out:
            out.write(ungapfasta[:-1])            
        with open(aafile, 'w+') as out:
            out.write(aafasta[:-1])            
            

prelimal = 'FCoV1_full_nt_ginsi_Izumi_250824.fasta'
project = 'FCoV1_144comp'
extract_ORFs(ORFs, prelimal, project, 'Y', 'Y')

/opt/anaconda3/envs/FCoV/lib/python3.13/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


ExtractしたORFをalignment　AAのfileと読み枠を考慮して塩基配列に戻したfileを作成

In [10]:
def ORFal_aa_to_nt(orfd, pr_name, mafft_path, p2n_path):
    
    for orf in list(orfd):
        
        aafile = '%s_%s_aa.fas'%(pr_name, orf)
        alaafile = 'al_%s_%s_aa.fas'%(pr_name, orf)
        ntoutfile = 'al_%s_%s_nt.fas'%(pr_name, orf)
        ntfile = '%s_%s_nt.fas'%(pr_name, orf)


        cmd = '%s --localpair --maxiterate 1000 %s > %s'%(mafft_path, aafile, alaafile)
        subprocess.Popen([cmd], shell = True,close_fds=True).wait()
        
        cmd = 'perl %s %s %s -output fasta > %s'%(p2n_path, alaafile, ntfile, ntoutfile)
        subprocess.Popen([cmd], shell = True,close_fds=True).wait()

mafft_path = 'mafft'
p2n_path = '/Applications/pal2nal_v14/pal2nal.pl'

ORFal_aa_to_nt(ORFs, project, mafft_path, p2n_path)

outputhat23=16
treein = 0
compacttree = 0
stacksize: 8176 kb
rescale = 1
All-to-all alignment.
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 
  140 / 144
done.

Progressive alignment ... 
STEP   110 /143 
Reallocating..done. *alloclen = 14781
STEP   132 /143 
Reallocating..done. *alloclen = 15990
STEP   137 /143 
Reallocating..done. *alloclen = 19776
STEP   143 /143 
done.
tbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
1 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 16
sueff_global = 0.100000
nadd = 16
Loading 'hat3' ... done.
rescale = 1

  140 / 144
Segment   1/  1    1-12868
STEP 005-096-1  identical.    identical

## Nやgapの多い配列をalignmentから除去する

In [12]:
import pandas as pd
from Bio import SeqIO

In [13]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [109]:
seq = fasta_to_dataframe("al_FCoV1_144comp_S_aa.fas")
seq

,ID,seq
0,KY566209.1,MILLLLAFFSVVKSQDAPHGVTLPQF---------NTSHNNDKFEL...
1,PP254206.1,MIVLILALLSTASSRDAPNGVTLPHF---------NTSYNNAKFEL...
2,PP254207.1,MIVLIFALLSTVRALDAPHGVXTPLF---------NTSNDNDKFEL...
3,PP254208.1,MIVLJXALLSTXSSNDAPXGXTLPHF---------NTXYNNAKFEL...
4,PP254209.1,MILLLXALLSVXSSEDAPHGAXLPHF---------NTSHNNHKFEL...
...,...,...
139,FECV47,MIVLILALLSTVRSEDAPHGVTLPQF---------NTSYNSQKFEL...
140,FECV5,MILLLLTFLTAVRSEDAPHGVTLPQF---------NTSHNNDKFEL...
141,FECV51,-MIVLLALLTVVRSEDAPHGAILPHF---------NTSHNNTKFEL...
142,FECV66,-MIVLLALLTVVRSEDAPHGAILPHF---------NTSHNNTKFEL...


各配列の中で、’N’と”ー”がいくつ含まれるのかを数える

In [110]:
# count how many 'N' and '-' in each seq and add as a new column
seq["N_count"] = seq["seq"].str.count("N")
seq["gap_count"] = seq["seq"].str.count("-")
seq

,ID,seq,N_count,gap_count
0,KY566209.1,MILLLLAFFSVVKSQDAPHGVTLPQF---------NTSHNNDKFEL...,112,1076
1,PP254206.1,MIVLILALLSTASSRDAPNGVTLPHF---------NTSYNNAKFEL...,104,1079
2,PP254207.1,MIVLIFALLSTVRALDAPHGVXTPLF---------NTSNDNDKFEL...,113,1077
3,PP254208.1,MIVLJXALLSTXSSNDAPXGXTLPHF---------NTXYNNAKFEL...,102,1076
4,PP254209.1,MILLLXALLSVXSSEDAPHGAXLPHF---------NTSHNNHKFEL...,111,1078
...,...,...,...,...
139,FECV47,MIVLILALLSTVRSEDAPHGVTLPQF---------NTSYNSQKFEL...,110,1078
140,FECV5,MILLLLTFLTAVRSEDAPHGVTLPQF---------NTSHNNDKFEL...,112,1079
141,FECV51,-MIVLLALLTVVRSEDAPHGAILPHF---------NTSHNNTKFEL...,116,1079
142,FECV66,-MIVLLALLTVVRSEDAPHGAILPHF---------NTSHNNTKFEL...,114,1079


In [111]:
# アライメントの長さを取得
# length of seq alignment
length_of_alignment = seq["seq"].str.len().iloc[0] # 最後の.iloc[0]は、一番最初の行っていう意味。今回の場合は、アライメント後の配列で、全ての配列長は同じなので、どれか一つを取れば良い
print(f"Length of alignment: {length_of_alignment}")

Length of alignment: 2545


全体のアライメントの長さのうち、半分以上が'N'もしくは'-'の配列を除く。以下で行なっている処理は、'N'の数が配列長 * 0.5以下、かつ'-'の数が配列長 * 0.5以下の行のみを選択しています

In [112]:
# remove rows if they contain 'N' or '-' more than 50% of the length of alignment
seq_clean = seq[(seq["N_count"] <= length_of_alignment * 0.5) & (seq["gap_count"] <= length_of_alignment * 0.5)]
seq_clean

,ID,seq,N_count,gap_count
0,KY566209.1,MILLLLAFFSVVKSQDAPHGVTLPQF---------NTSHNNDKFEL...,112,1076
1,PP254206.1,MIVLILALLSTASSRDAPNGVTLPHF---------NTSYNNAKFEL...,104,1079
2,PP254207.1,MIVLIFALLSTVRALDAPHGVXTPLF---------NTSNDNDKFEL...,113,1077
3,PP254208.1,MIVLJXALLSTXSSNDAPXGXTLPHF---------NTXYNNAKFEL...,102,1076
4,PP254209.1,MILLLXALLSVXSSEDAPHGAXLPHF---------NTSHNNHKFEL...,111,1078
...,...,...,...,...
139,FECV47,MIVLILALLSTVRSEDAPHGVTLPQF---------NTSYNSQKFEL...,110,1078
140,FECV5,MILLLLTFLTAVRSEDAPHGVTLPQF---------NTSHNNDKFEL...,112,1079
141,FECV51,-MIVLLALLTVVRSEDAPHGAILPHF---------NTSHNNTKFEL...,116,1079
142,FECV66,-MIVLLALLTVVRSEDAPHGAILPHF---------NTSHNNTKFEL...,114,1079


ほとんどNみたいな配列を除いて、綺麗になったデータフレームから、IDとseqのカラムを使って、fasta形式で出力

In [113]:
# export as fasta
with open("al_FCoV1_144comp_S_aa_cleaned.fas", "w") as f:
    for index, row in seq_clean.iterrows():
        f.write(f">{row['ID']}\n{row['seq']}\n")